# 08 — ML clustering


In [1]:
from pathlib import Path
import sys

import pandas as pd

# ---------------------------------------------------------
# Resolve project root
# ---------------------------------------------------------
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


Project root: /Users/subhankarbiswas/smart-city-knowledge-graph-v3


In [2]:
from src.clustering import (
    build_profile_matrix,
    cluster_kmeans,
    cluster_profiles,
    cluster_dbscan,
)

print("Clustering functions imported successfully.")

Clustering functions imported successfully.


In [3]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

ACCESSIBILITY_PATH = (
    PROCESSED_DIR / "network_accessibility.csv"
)

PROFILE_PATH = (
    PROCESSED_DIR / "accessibility_profiles.csv"
)

KMEANS_PROFILE_PATH = (
    PROCESSED_DIR / "cluster_profiles.csv"
)

DBSCAN_PROFILE_PATH = (
    PROCESSED_DIR / "dbscan_cluster_profiles.csv"
)

METRICS_PATH = (
    PROCESSED_DIR / "clustering_metrics.csv"
)

# ---------------------------------------------------------
# Clustering configuration
# ---------------------------------------------------------
KMEANS_K = 4

print(f"K-Means clusters: {KMEANS_K}")
print(f"Input: {ACCESSIBILITY_PATH}")

K-Means clusters: 4
Input: /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/network_accessibility.csv


In [4]:
if not ACCESSIBILITY_PATH.exists():
    raise FileNotFoundError(
        "network_accessibility.csv was not found:\n"
        f"{ACCESSIBILITY_PATH}\n\n"
        "Run Notebook 06 first."
    )

raw = pd.read_csv(ACCESSIBILITY_PATH)

print("=" * 60)
print("NETWORK ACCESSIBILITY DATA")
print("=" * 60)

print(f"Rows:    {len(raw):,}")
print(f"Columns: {len(raw.columns):,}")

display(raw.head())

NETWORK ACCESSIBILITY DATA
Rows:    110
Columns: 5


,origin_index,network_node,travel_time_s,travel_time_min,service
0,6,5144049364,1933.581874,32.226365,hospital
1,7,428473989,2343.797150,39.063286,hospital
2,8,428469705,2151.900526,35.865009,hospital
3,9,428473989,2343.797150,39.063286,hospital
4,10,254161663,715.177882,11.919631,hospital


In [5]:
REQUIRED_COLUMNS = [
    "origin_index",
    "travel_time_min",
    "service",
]

missing_columns = [
    column
    for column in REQUIRED_COLUMNS
    if column not in raw.columns
]

if missing_columns:
    raise KeyError(
        "Required columns are missing from "
        "network_accessibility.csv:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )

print("Required columns found:")
for column in REQUIRED_COLUMNS:
    print(f"  ✓ {column}")

Required columns found:
  ✓ origin_index
  ✓ travel_time_min
  ✓ service


In [6]:
raw = raw.copy()

raw["service"] = (
    raw["service"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

raw["origin_index"] = pd.to_numeric(
    raw["origin_index"],
    errors="coerce",
)

raw["travel_time_min"] = pd.to_numeric(
    raw["travel_time_min"],
    errors="coerce",
)

before = len(raw)

raw = raw.dropna(
    subset=[
        "origin_index",
        "travel_time_min",
        "service",
    ]
).copy()

raw = raw[
    raw["service"].astype(bool)
].copy()

after = len(raw)

print("=" * 60)
print("CLEANING")
print("=" * 60)

print(f"Rows before cleaning: {before:,}")
print(f"Rows after cleaning:  {after:,}")
print(f"Rows removed:        {before - after:,}")

CLEANING
Rows before cleaning: 110
Rows after cleaning:  110
Rows removed:        0


In [7]:
service_counts = (
    raw["service"]
    .value_counts()
    .rename_axis("service")
    .reset_index(name="records")
)

print("=" * 60)
print("SERVICES AVAILABLE FOR CLUSTERING")
print("=" * 60)

display(service_counts)

SERVICES AVAILABLE FOR CLUSTERING


,service,records
0,hospital,22
1,pharmacy,22
2,supermarket,22
3,train_station,22
4,library,22


In [8]:
access = {
    s: g[["origin_index", "travel_time_min"]]
    for s, g in raw.groupby("service")
}

In [9]:
access = {}

for service, group in raw.groupby("service", sort=True):

    service_data = group[
        [
            "origin_index",
            "travel_time_min",
        ]
    ].copy()

    service_data = service_data.sort_values(
        "origin_index"
    )

    access[service] = service_data

    print(
        f"{service:<20} "
        f"{len(service_data):,} records"
    )

hospital             22 records
library              22 records
pharmacy             22 records
supermarket          22 records
train_station        22 records


In [10]:
profile = build_profile_matrix(access)

if profile is None or profile.empty:
    raise ValueError(
        "build_profile_matrix() returned an empty profile matrix."
    )

profile = profile.copy()

print("=" * 60)
print("ACCESSIBILITY PROFILE MATRIX")
print("=" * 60)

print(f"Origins:  {len(profile):,}")
print(f"Features: {len(profile.columns):,}")

display(profile.head())

ACCESSIBILITY PROFILE MATRIX
Origins:  22
Features: 5


,hospital,library,pharmacy,supermarket,train_station
origin_index,,,,,
6,32.226365,32.184804,21.226137,7.849504,22.478618
7,39.063286,34.455787,20.708838,14.051300,24.140200
8,35.865009,31.257510,17.510561,10.853023,20.941922
9,39.063286,34.455787,20.708838,14.051300,24.140200
10,11.919631,5.663331,4.256117,1.382590,6.745427


In [11]:
print("=" * 60)
print("PROFILE MATRIX VALIDATION")
print("=" * 60)

print("Missing values:")
display(
    profile.isna()
    .sum()
    .rename("missing")
    .to_frame()
)

print("\nData types:")
display(
    profile.dtypes
    .rename("dtype")
    .to_frame()
)

PROFILE MATRIX VALIDATION
Missing values:


,missing
hospital,0
library,0
pharmacy,0
supermarket,0
train_station,0



Data types:


,dtype
hospital,float64
library,float64
pharmacy,float64
supermarket,float64
train_station,float64


In [12]:
numeric_profile = profile.select_dtypes(
    include="number"
).copy()

if numeric_profile.empty:
    raise ValueError(
        "No numeric features are available for clustering."
    )

numeric_profile = numeric_profile.replace(
    [float("inf"), float("-inf")],
    pd.NA,
)

if numeric_profile.isna().any().any():
    print(
        "Warning: missing/non-finite values detected "
        "in clustering features."
    )
else:
    print("✓ Clustering matrix contains no missing/non-finite values.")

✓ Clustering matrix contains no missing/non-finite values.


In [13]:
if KMEANS_K < 2:
    raise ValueError(
        "K-Means requires K >= 2."
    )

if KMEANS_K >= len(profile):
    raise ValueError(
        f"K={KMEANS_K} must be smaller than the "
        f"number of origins ({len(profile)})."
    )

labels, metrics = cluster_kmeans(
    profile,
    k=KMEANS_K,
)

print("=" * 60)
print("K-MEANS RESULTS")
print("=" * 60)

print("Metrics:")
print(metrics)

K-MEANS RESULTS
Metrics:
{'algorithm': 'KMeans', 'k': 4, 'silhouette': 0.422984485102276, 'davies_bouldin': 0.9113635797757208, 'calinski_harabasz': 24.827347841526613}


In [14]:
profile_kmeans = profile.copy()

profile_kmeans["cluster_kmeans"] = labels

print("=" * 60)
print("K-MEANS CLUSTER DISTRIBUTION")
print("=" * 60)

cluster_counts = (
    profile_kmeans["cluster_kmeans"]
    .value_counts()
    .sort_index()
    .rename_axis("cluster")
    .reset_index(name="origins")
)

display(cluster_counts)

K-MEANS CLUSTER DISTRIBUTION


,cluster,origins
0,0,3
1,1,7
2,2,6
3,3,6


In [15]:
kmeans_cluster_profiles = cluster_profiles(
    profile_kmeans.drop(
        columns=["cluster_kmeans"]
    ),
    labels,
)

if kmeans_cluster_profiles is None:
    raise ValueError(
        "cluster_profiles() returned None."
    )

print("=" * 60)
print("K-MEANS CLUSTER PROFILES")
print("=" * 60)

display(kmeans_cluster_profiles)

K-MEANS CLUSTER PROFILES


,hospital,library,pharmacy,supermarket,train_station
cluster,,,,,
0,39.004195,36.853875,24.742199,0.000000,27.150633
1,12.173986,8.537241,4.052821,3.027446,7.614627
2,35.049666,32.683307,20.331164,11.760913,22.732488
3,19.866912,18.545001,9.654002,2.985786,18.284109


In [16]:
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

profile_kmeans.to_csv(
    PROFILE_PATH,
    index=False
)

kmeans_cluster_profiles.to_csv(
    KMEANS_PROFILE_PATH,
    index=False
)

print("Saved:")
print(f"  ✓ {PROFILE_PATH}")
print(f"  ✓ {KMEANS_PROFILE_PATH}")

Saved:
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/accessibility_profiles.csv
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/cluster_profiles.csv


In [17]:
dbscan_input = profile_kmeans.drop(
    columns=["cluster_kmeans"]
).copy()

db_labels, db_metrics = cluster_dbscan(
    dbscan_input
)

print("=" * 60)
print("DBSCAN RESULTS")
print("=" * 60)

print("Metrics:")
print(db_metrics)

DBSCAN RESULTS
Metrics:
{'algorithm': 'DBSCAN', 'eps': 0.8, 'min_samples': 8}


In [18]:
profile_dbscan = profile.copy()

profile_dbscan["cluster_dbscan"] = db_labels

dbscan_distribution = (
    profile_dbscan["cluster_dbscan"]
    .value_counts()
    .sort_index()
    .rename_axis("cluster")
    .reset_index(name="origins")
)

print("=" * 60)
print("DBSCAN CLUSTER DISTRIBUTION")
print("=" * 60)

display(dbscan_distribution)

DBSCAN CLUSTER DISTRIBUTION


,cluster,origins
0,-1,22


In [19]:
dbscan_cluster_profiles = cluster_profiles(
    profile_dbscan.drop(
        columns=["cluster_dbscan"]
    ),
    db_labels,
)

print("=" * 60)
print("DBSCAN CLUSTER PROFILES")
print("=" * 60)

display(dbscan_cluster_profiles)

DBSCAN CLUSTER PROFILES


,hospital,library,pharmacy,supermarket,train_station
cluster,,,,,
-1,24.169543,21.71328,12.841243,4.985105,17.31154


In [20]:
profile_dbscan.to_csv(
    DBSCAN_PROFILE_PATH,
    index=False
)

dbscan_cluster_profiles.to_csv(
    DBSCAN_PROFILE_PATH,
    index=False
)

In [21]:
DBSCAN_ASSIGNMENTS_PATH = (
    PROCESSED_DIR / "accessibility_profiles_dbscan.csv"
)

DBSCAN_CLUSTER_PROFILE_PATH = (
    PROCESSED_DIR / "dbscan_cluster_profiles.csv"
)

profile_dbscan.to_csv(
    DBSCAN_ASSIGNMENTS_PATH,
    index=False
)

dbscan_cluster_profiles.to_csv(
    DBSCAN_CLUSTER_PROFILE_PATH,
    index=False
)

print("Saved DBSCAN results:")
print(f"  ✓ {DBSCAN_ASSIGNMENTS_PATH}")
print(f"  ✓ {DBSCAN_CLUSTER_PROFILE_PATH}")

Saved DBSCAN results:
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/accessibility_profiles_dbscan.csv
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/dbscan_cluster_profiles.csv


In [22]:
metric_rows = []

if isinstance(metrics, dict):
    kmeans_metric_row = {
        "algorithm": "KMeans",
        "k": KMEANS_K,
        **metrics,
    }
else:
    kmeans_metric_row = {
        "algorithm": "KMeans",
        "k": KMEANS_K,
        "metrics": str(metrics),
    }

metric_rows.append(kmeans_metric_row)

if isinstance(db_metrics, dict):
    dbscan_metric_row = {
        "algorithm": "DBSCAN",
        **db_metrics,
    }
else:
    dbscan_metric_row = {
        "algorithm": "DBSCAN",
        "metrics": str(db_metrics),
    }

metric_rows.append(dbscan_metric_row)

metrics_df = pd.DataFrame(metric_rows)

display(metrics_df)

,algorithm,k,silhouette,davies_bouldin,calinski_harabasz,eps,min_samples
0,KMeans,4.0,0.422984,0.911364,24.827348,NaN,NaN
1,DBSCAN,NaN,NaN,NaN,NaN,0.8,8.0


In [23]:
metrics_df.to_csv(
    METRICS_PATH,
    index=False
)

print(f"Saved clustering metrics to:")
print(METRICS_PATH)

Saved clustering metrics to:
/Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/clustering_metrics.csv


In [24]:
print("=" * 60)
print("NOTEBOOK 08 VALIDATION")
print("=" * 60)

print(f"Accessibility records: {len(raw):,}")
print(f"Origins profiled:      {len(profile):,}")
print(f"Profile features:      {len(profile.columns):,}")

print("\nK-Means:")
print(f"  K: {KMEANS_K}")
print(
    f"  Clusters: "
    f"{profile_kmeans['cluster_kmeans'].nunique()}"
)

print("\nDBSCAN:")
print(
    f"  Labels: "
    f"{profile_dbscan['cluster_dbscan'].nunique()}"
)

print("\nOutput files:")

output_files = [
    PROFILE_PATH,
    KMEANS_PROFILE_PATH,
    DBSCAN_ASSIGNMENTS_PATH,
    DBSCAN_CLUSTER_PROFILE_PATH,
    METRICS_PATH,
]

for path in output_files:
    if path.exists():
        print(f"  ✓ {path}")
    else:
        print(f"  ✗ {path}")

NOTEBOOK 08 VALIDATION
Accessibility records: 110
Origins profiled:      22
Profile features:      5

K-Means:
  K: 4
  Clusters: 4

DBSCAN:
  Labels: 1

Output files:
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/accessibility_profiles.csv
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/cluster_profiles.csv
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/accessibility_profiles_dbscan.csv
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/dbscan_cluster_profiles.csv
  ✓ /Users/subhankarbiswas/smart-city-knowledge-graph-v3/data/processed/clustering_metrics.csv
